# Music Annotation Project V7.9: Observed-Ink / Glyph-Preservation Challenger

**Runtime:** CPU. Select **Runtime > Run all**.

This is a controlled development benchmark. It compares new observed-ink preservation routes with the frozen V7.8 `geometry_t80` baseline.

## Hard guardrails

- Notebook 09 is not read or modified.
- The frozen V7.8 independent input, output and outcome directories are prohibited.
- Only the existing controlled development pages are used.
- Clean pages are used only for post-reconstruction labels and scoring.
- Candidate routing uses only the observed page and controlled removal mask.
- Dilation 2, Telea, paper estimation, remove semantics and ambiguous semantics remain frozen initially.
- Pixels outside the removal mask must remain byte-for-byte unchanged.
- V7.8 baseline results are recreated and checked against the durable V7.5 cache before challenger comparisons.
- This notebook develops reconstruction challengers only. It does not train or apply a structural auto-safe policy.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib scikit-learn


## Configuration and fail-closed data boundary


In [ ]:
import cv2, json, hashlib, math, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from sklearn.metrics import precision_recall_fscore_support

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
OUTPUT_ROOT=ROOT/'observed_ink_preservation_challenger_v7_9'
V75_RUN_OVERRIDE=None
PAGE_ALLOWLIST=None
MAX_PAGES=None
FORCE_REBUILD=False

MASK_DILATION=2
CROP_MARGIN=48
MIN_COMPONENT_AREA=20
GEOMETRY_THRESHOLD=80
BAND_WIDTH=4
CONTEXT_RING_RADIUS=18

# Development-only search grids. These are selected using page-grouped development evidence.
DARK_THRESHOLDS=[80,100,120,140,160]
EDGE_THRESHOLDS=[20,35,50]
STROKE_WIDTH_LIMITS=[2.5,3.5,5.0]
MIN_GLYPH_COMPONENT_AREAS=[3,6,12]
MAX_GLYPH_COMPONENT_AREAS=[650,1200,2400]

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
PROHIBITED_PATH_TOKENS=[
 'independent_structural_validation_v7_8_builder',
 'independent_structural_validation_v7_8_rebuilt',
 'independent_policy_validation_v7_8',
]

def assert_development_path(path):
    text=str(Path(path)).lower()
    if any(token in text for token in PROHIBITED_PATH_TOKENS):
        raise RuntimeError(f'PROHIBITED V7.8 independent path accessed: {path}')

def sha256_file(path):
    assert_development_path(path)
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()

def latest_complete(root,required):
    assert_development_path(root)
    hits=[]
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest_complete(V75_ROOT,['tables/cache_status.csv','tables/surrounding_area_evidence_by_component.csv'])
if V75 is None:raise RuntimeError('No complete controlled V7.5 development run found.')
assert_development_path(V75)
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT=OUTPUT_ROOT/RUN_NAME
TABLES=OUT/'tables';CHARTS=OUT/'charts';CONTACTS=OUT/'contact_sheets';CACHE=OUT/'cache';DIFFS=OUT/'difference_images'
for d in [OUT,TABLES,CHARTS,CONTACTS,CACHE,DIFFS]:d.mkdir(parents=True,exist_ok=False)
print('Controlled V7.5 reference:',V75)
print('Output:',OUT)


## Frozen V7.8 baseline functions

These functions reproduce the existing dilation-2, Telea and `geometry_t80` baseline. The challenger does not edit them.


In [ ]:
def immutable(original,candidate,mask):
    out=original.copy();out[mask>0]=candidate[mask>0]
    assert np.array_equal(out[mask==0],original[mask==0])
    return out

def background_surface(observed,mask,sigma=45.0):
    visible=(mask==0)&(observed>=200);values=observed.astype(np.float32)*visible;weights=visible.astype(np.float32)
    num=cv2.GaussianBlur(values,(0,0),sigma);den=cv2.GaussianBlur(weights,(0,0),sigma);surface=num/np.maximum(den,1e-4);fallback=cv2.GaussianBlur(observed,(0,0),sigma)
    return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)

def support_maps_v78(observed,mask):
    dark=(observed<190)&(mask==0)
    horizontal=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    vertical=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
    n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);compact=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,h,a=stats[i]
        if 3<=a<=650 and w<=40 and h<=40:compact[lab==i]=True
    k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8)
    return cv2.dilate(horizontal.astype(np.uint8),k)>0,cv2.dilate(vertical.astype(np.uint8),k)>0,cv2.dilate(compact.astype(np.uint8),k)>0

def geometry_t80_v78(observed,mask,telea):
    paper=background_surface(observed,mask);normalised=cv2.divide(observed,np.maximum(paper,1),scale=255)
    horizontal,vertical,compact=support_maps_v78(observed,mask);m=mask>0
    preserve=m&(horizontal|vertical|compact)&(observed<200)
    remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(normalised>=GEOMETRY_THRESHOLD)
    ambiguous=m&~preserve&~remove
    result=observed.copy();result[remove]=paper[remove];result[ambiguous]=telea[ambiguous];result[preserve]=observed[preserve]
    return immutable(observed,result,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact,'glyph':np.zeros_like(m),'paper':paper,'normalised':normalised}


## V7.9 observed-ink preservation candidates

The candidates add conservative glyph-like observable-ink support. They never use clean truth during reconstruction.

- `connected_dark`: preserves bounded dark connected structures visible inside the mask when those structures connect to a narrow boundary band.
- `edge_stroke`: adds dark high-edge pixels with locally bounded stroke width.
- `consensus`: requires both connected-dark and edge/stroke evidence.
- `union`: accepts either evidence route and is intentionally the least conservative development candidate.


In [ ]:
def mask_boundary_band(mask,width=3):
    m=(mask>0).astype(np.uint8);inner=cv2.erode(m,np.ones((2*width+1,2*width+1),np.uint8))>0
    return (m>0)&~inner

def bounded_inside_components(observed,mask,dark_threshold,min_area,max_area,max_width=80,max_height=80):
    dark=(observed<dark_threshold)&(mask>0);n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8)
    support=np.zeros_like(dark);boundary=mask_boundary_band(mask,3)
    for i in range(1,n):
        x,y,w,h,a=stats[i]
        comp=lab==i
        if min_area<=a<=max_area and w<=max_width and h<=max_height and np.any(comp&boundary):support[comp]=True
    return support

def stroke_width_proxy(binary):
    return cv2.distanceTransform(binary.astype(np.uint8),cv2.DIST_L2,5)

def glyph_support(observed,mask,params,mode):
    connected=bounded_inside_components(observed,mask,params['dark_threshold'],params['min_area'],params['max_area'])
    edges=cv2.Canny(observed,70,170)>0
    dark=(observed<params['dark_threshold'])&(mask>0)
    width=stroke_width_proxy(dark)
    edge_stroke=dark&(cv2.dilate(edges.astype(np.uint8),np.ones((3,3),np.uint8))>0)&(width<=params['stroke_width_limit'])
    if mode=='connected_dark':raw=connected
    elif mode=='edge_stroke':raw=edge_stroke
    elif mode=='consensus':raw=connected&edge_stroke
    elif mode=='union':raw=connected|edge_stroke
    else:raise ValueError(mode)
    # Avoid isolated single-pixel preservation noise.
    n,lab,stats,_=cv2.connectedComponentsWithStats(raw.astype(np.uint8),8);clean=np.zeros_like(raw)
    for i in range(1,n):
        if stats[i,cv2.CC_STAT_AREA]>=params['min_area']:clean[lab==i]=True
    return clean

def geometry_v79(observed,mask,telea,params,mode):
    paper=background_surface(observed,mask);normalised=cv2.divide(observed,np.maximum(paper,1),scale=255)
    horizontal,vertical,compact=support_maps_v78(observed,mask);glyph=glyph_support(observed,mask,params,mode);m=mask>0
    preserve=m&(horizontal|vertical|compact|glyph)&(observed<200)
    remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(normalised>=GEOMETRY_THRESHOLD)
    ambiguous=m&~preserve&~remove
    result=observed.copy();result[remove]=paper[remove];result[ambiguous]=telea[ambiguous];result[preserve]=observed[preserve]
    return immutable(observed,result,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact,'glyph':glyph,'paper':paper,'normalised':normalised}


## Load controlled development pages and verify the frozen baseline


In [ ]:
def image_map(folder):
    assert_development_path(folder)
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def loadz(path):
    assert_development_path(path)
    with np.load(path) as z:return {k:z[k] for k in z.files}

clean_map=image_map(ROOT/'source'/'clean');observed_map=image_map(ROOT/'aligned')
status=pd.read_csv(V75/'tables'/'cache_status.csv');cache_lookup={str(r.page):Path(r.cache) for _,r in status.iterrows()}
pages=sorted(set(clean_map)&set(observed_map)&set(cache_lookup))
if PAGE_ALLOWLIST is not None:pages=[p for p in pages if p in set(PAGE_ALLOWLIST)]
if MAX_PAGES is not None:pages=pages[:MAX_PAGES]
if not pages:raise RuntimeError('Controlled development roster is empty.')

baseline_checks=[];development={}
for page in pages:
    z=loadz(cache_lookup[page]);clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0)
    required=['mask','telea','geometry_t80']
    missing=[x for x in required if x not in z]
    if missing:raise RuntimeError(f'{page}: missing cached arrays {missing}')
    mask=z['mask'];telea=z['telea'];rebuilt,routes=geometry_t80_v78(observed,mask,telea)
    baseline_checks.append({'page':page,'geometry_exact':bool(np.array_equal(rebuilt,z['geometry_t80'])),'different_pixels':int(np.sum(rebuilt!=z['geometry_t80'])),'outside_immutable':bool(np.array_equal(rebuilt[mask==0],observed[mask==0]))})
    development[page]={'clean':clean,'observed':observed,'mask':mask,'telea':telea,'baseline':rebuilt,'baseline_routes':routes}
baseline_df=pd.DataFrame(baseline_checks);baseline_df.to_csv(TABLES/'baseline_equivalence.csv',index=False)
if not baseline_df.geometry_exact.all() or not baseline_df.outside_immutable.all():raise RuntimeError('Frozen V7.8 baseline does not reproduce the durable V7.5 development cache.')
print('Frozen V7.8 baseline reproduced on',len(pages),'development pages.')


## Clean-truth labels and evaluation metrics

Clean truth is introduced only here, after every candidate output has been generated.


In [ ]:
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im):return cv2.Canny(im,70,170)>0
def tolerant_scores(ref,pred,region,tol=1):
    ref=ref&region;pred=pred&region
    if not ref.sum():return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8);rd=cv2.dilate(ref.astype(np.uint8),k)>0;pd=cv2.dilate(pred.astype(np.uint8),k)>0
    precision=float(np.sum(pred&rd)/max(pred.sum(),1));recall=float(np.sum(ref&pd)/ref.sum());return precision,recall,float(2*precision*recall/max(precision+recall,1e-12))

def score_candidate(clean,observed,recon,mask,routes):
    m=mask>0;ci=printed_ink(clean);ri=printed_ink(recon)
    ip,ir,if1=tolerant_scores(ci,ri,m);ep,er,ef1=tolerant_scores(edges(clean),edges(recon),m)
    diff=np.abs(clean.astype(np.float32)-recon.astype(np.float32));preserve=routes['preserve']&m;glyph=routes['glyph']&m
    # False preservation: preserved observed ink that clean truth says is not printed ink.
    false_preserved=int(np.sum(preserve&~ci));true_preserved=int(np.sum(preserve&ci))
    glyph_false=int(np.sum(glyph&~ci));glyph_true=int(np.sum(glyph&ci))
    return {'mask_pixels':int(m.sum()),'mae':float(diff[m].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,'preserve_pixels':int(preserve.sum()),'true_print_preserved':true_preserved,'false_annotation_preserved':false_preserved,'preserve_precision':true_preserved/max(int(preserve.sum()),1),'glyph_pixels':int(glyph.sum()),'glyph_true_print':glyph_true,'glyph_false_annotation':glyph_false,'glyph_precision':glyph_true/max(int(glyph.sum()),1),'removed_clean_print_fraction':float(np.sum(routes['remove']&ci&m)/max(m.sum(),1)),'outside_mask_drift':int(np.sum(recon[~m]!=observed[~m]))}


## Run baseline and challenger grid


In [ ]:
MODES=['connected_dark','edge_stroke','consensus','union']
configs=[]
for dt in DARK_THRESHOLDS:
 for sw in STROKE_WIDTH_LIMITS:
  for mina in MIN_GLYPH_COMPONENT_AREAS:
   for maxa in MAX_GLYPH_COMPONENT_AREAS:
    if mina<=maxa:configs.append({'dark_threshold':dt,'stroke_width_limit':sw,'min_area':mina,'max_area':maxa})

rows=[];component_rows=[];recon_cache={}
for page,z in development.items():
    clean=z['clean'];observed=z['observed'];mask=z['mask'];telea=z['telea']
    candidates={'v78_baseline':(z['baseline'],z['baseline_routes'])}
    for mode in MODES:
        for params in configs:
            name=f"v79_{mode}_d{params['dark_threshold']}_w{params['stroke_width_limit']}_a{params['min_area']}_{params['max_area']}"
            candidates[name]=geometry_v79(observed,mask,telea,params,mode)
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    for name,(recon,routes) in candidates.items():
        q=score_candidate(clean,observed,recon,mask,routes);rows.append({'page':page,'candidate':name,**q})
        if q['outside_mask_drift']!=0:raise RuntimeError(f'{page} {name}: outside-mask drift')
        for cid in range(1,n):
            if stats[cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
            comp=lab==cid;ci=printed_ink(clean);ri=printed_ink(recon);ip,ir,if1=tolerant_scores(ci,ri,comp);ep,er,ef1=tolerant_scores(edges(clean),edges(recon),comp)
            component_rows.append({'page':page,'component':cid,'candidate':name,'area':int(comp.sum()),'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,'mae':float(np.abs(clean.astype(np.float32)-recon.astype(np.float32))[comp].mean()),'glyph_fraction':float(np.sum(routes['glyph']&comp)/comp.sum()),'preserve_fraction':float(np.sum(routes['preserve']&comp)/comp.sum()),'false_annotation_preserved':int(np.sum(routes['preserve']&comp&~ci))})
        recon_cache[(page,name)]=(recon,routes)
    print('Benchmarked',page,'candidates',len(candidates))
results=pd.DataFrame(rows);components=pd.DataFrame(component_rows)
results.to_csv(TABLES/'candidate_metrics_by_page.csv',index=False);components.to_csv(TABLES/'candidate_metrics_by_component.csv',index=False)
print('Candidate configurations:',results.candidate.nunique())


## Page-grouped challenger selection

Selection is based on development pages only. The score rewards printed-ink and edge preservation but applies a dominant penalty to preserved annotation pixels. The final ranking is accompanied by every underlying metric rather than relying on the composite alone.


In [ ]:
summary=results.groupby('candidate',as_index=False).agg(pages=('page','nunique'),ink_recall=('ink_recall','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),mae=('mae','mean'),preserve_precision=('preserve_precision','mean'),glyph_precision=('glyph_precision','mean'),false_annotation_preserved=('false_annotation_preserved','sum'),true_print_preserved=('true_print_preserved','sum'),removed_clean_print_fraction=('removed_clean_print_fraction','mean'))
summary['utility']=summary['ink_recall'].fillna(1)+summary['edge_f1'].fillna(1)+summary['preserve_precision'].fillna(1)-10*(summary.false_annotation_preserved/(summary.true_print_preserved+summary.false_annotation_preserved).clip(lower=1))-summary.removed_clean_print_fraction*20-summary.mae/255
base=summary[summary.candidate=='v78_baseline'].iloc[0]
summary['delta_ink_recall']=summary.ink_recall-base.ink_recall;summary['delta_edge_f1']=summary.edge_f1-base.edge_f1;summary['delta_mae']=summary.mae-base.mae;summary['delta_false_annotation_preserved']=summary.false_annotation_preserved-base.false_annotation_preserved
summary=summary.sort_values(['false_annotation_preserved','utility'],ascending=[True,False]);summary.to_csv(TABLES/'candidate_summary.csv',index=False)

# Hard development eligibility: no increase in total false annotation preservation and improvement in ink recall or edge F1.
eligible=summary[(summary.candidate!='v78_baseline')&(summary.delta_false_annotation_preserved<=0)&((summary.delta_ink_recall>0)|(summary.delta_edge_f1>0))].copy()
if eligible.empty:
    selected=None;print('NO V7.9 CHALLENGER MET THE CONTROLLED DEVELOPMENT ELIGIBILITY RULE.')
else:
    selected=eligible.sort_values('utility',ascending=False).iloc[0];print('Selected development challenger:',selected.candidate)
display(summary.head(30))


## Paired page and component comparisons


In [ ]:
if selected is not None:
    candidate=selected.candidate
    paired=results[results.candidate.isin(['v78_baseline',candidate])].pivot(index='page',columns='candidate',values=['ink_recall','ink_f1','edge_f1','mae','false_annotation_preserved','true_print_preserved','removed_clean_print_fraction'])
    deltas=[]
    for metric in ['ink_recall','ink_f1','edge_f1','mae','false_annotation_preserved','true_print_preserved','removed_clean_print_fraction']:
        d=paired[metric][candidate]-paired[metric]['v78_baseline'];deltas.append({'metric':metric,'mean_delta':float(d.mean()),'median_delta':float(d.median()),'wins':int((d>0).sum()),'losses':int((d<0).sum()),'ties':int((d==0).sum())})
    paired.to_csv(TABLES/'selected_challenger_paired_pages.csv');pd.DataFrame(deltas).to_csv(TABLES/'selected_challenger_page_deltas.csv',index=False);display(pd.DataFrame(deltas))


## Diagnostic failure and improvement crops


In [ ]:
def save_triptych(page,cid,candidate,kind,rank):
    z=development[page];clean=z['clean'];observed=z['observed'];base=z['baseline'];challenger=recon_cache[(page,candidate)][0]
    mask=z['mask']>0;n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8);x,y,w,h=stats[int(cid),:4];m=55;x0=max(0,x-m);y0=max(0,y-m);x1=min(clean.shape[1],x+w+m);y1=min(clean.shape[0],y+h+m)
    panels=[clean[y0:y1,x0:x1],observed[y0:y1,x0:x1],base[y0:y1,x0:x1],challenger[y0:y1,x0:x1]];names=['clean truth','observed','frozen V7.8','V7.9 challenger']
    fig,axes=plt.subplots(1,4,figsize=(18,5))
    for ax,im,name in zip(axes,panels,names):ax.imshow(im,cmap='gray',vmin=0,vmax=255);ax.set_title(name);ax.axis('off')
    fig.suptitle(f'{kind} #{rank} | {page} component {cid}');plt.tight_layout();folder=CONTACTS/kind;folder.mkdir(parents=True,exist_ok=True);plt.savefig(folder/f'{rank:03d}_{page}_component_{int(cid):04d}.png',dpi=170,bbox_inches='tight');plt.close(fig)

if selected is not None:
    candidate=selected.candidate
    c=components[components.candidate.isin(['v78_baseline',candidate])].pivot(index=['page','component'],columns='candidate',values=['ink_recall','edge_f1','mae','false_annotation_preserved']).reset_index()
    c['ink_gain']=c[('ink_recall',candidate)]-c[('ink_recall','v78_baseline')];c['edge_gain']=c[('edge_f1',candidate)]-c[('edge_f1','v78_baseline')];c['false_preserve_increase']=c[('false_annotation_preserved',candidate)]-c[('false_annotation_preserved','v78_baseline')]
    for rank,(_,r) in enumerate(c.sort_values('ink_gain',ascending=False).head(30).iterrows(),1):save_triptych(str(r['page'].iloc[0] if hasattr(r['page'],'iloc') else r['page']),int(r['component'].iloc[0] if hasattr(r['component'],'iloc') else r['component']),candidate,'largest_ink_recall_gains',rank)
    for rank,(_,r) in enumerate(c.sort_values('false_preserve_increase',ascending=False).head(30).iterrows(),1):save_triptych(str(r['page'].iloc[0] if hasattr(r['page'],'iloc') else r['page']),int(r['component'].iloc[0] if hasattr(r['component'],'iloc') else r['component']),candidate,'largest_false_preservation_increases',rank)


## Export manifest and decision record


In [ ]:
manifest={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'version':'7.9-controlled-observed-ink-preservation-challenger-1',
 'development_reference':str(V75),
 'development_pages':pages,
 'clean_truth_role':'post-reconstruction labels and scoring only',
 'v7_8_independent_data_read':False,
 'notebook_09_read_or_modified':False,
 'frozen_semantics':{'mask_dilation':MASK_DILATION,'crop_margin':CROP_MARGIN,'geometry_threshold':GEOMETRY_THRESHOLD,'band_width':BAND_WIDTH,'telea_unchanged':True,'paper_estimation_unchanged':True,'remove_rule_unchanged':True,'ambiguous_rule_unchanged':True},
 'baseline_equivalence_passed':bool(baseline_df.geometry_exact.all()),
 'candidate_modes':MODES,
 'parameter_grid':{'dark_thresholds':DARK_THRESHOLDS,'stroke_width_limits':STROKE_WIDTH_LIMITS,'minimum_areas':MIN_GLYPH_COMPONENT_AREAS,'maximum_areas':MAX_GLYPH_COMPONENT_AREAS},
 'candidate_count':int(results.candidate.nunique()-1),
 'selected_development_challenger':None if selected is None else str(selected.candidate),
 'selection_is_production_authorisation':False,
 'requires_new_structural_policy':True,
 'requires_new_untouched_independent_validation':True,
 'outputs':{'page_metrics':str(TABLES/'candidate_metrics_by_page.csv'),'component_metrics':str(TABLES/'candidate_metrics_by_component.csv'),'summary':str(TABLES/'candidate_summary.csv'),'contact_sheets':str(CONTACTS)},
}
(OUT/'v7_9_challenger_manifest.json').write_text(json.dumps(manifest,indent=2))
lines=[
 'V7.9 OBSERVED-INK / GLYPH-PRESERVATION CHALLENGER',
 '',
 f'Development pages: {len(pages)}',
 f'Candidates benchmarked: {manifest["candidate_count"]}',
 f'Baseline equivalence: {manifest["baseline_equivalence_passed"]}',
 f'Selected development challenger: {manifest["selected_development_challenger"]}',
 '',
 'This is development evidence only. Notebook 09 remains unchanged. V7.8 independent outcomes were not read. Any promoted V7.9 reconstruction requires a new structural-policy calibration and new untouched independent validation.'
]
report=chr(10).join(lines)
(OUT/'executive_summary.txt').write_text(report)
print(report)
print('Output:',OUT)
